# 🔍 Explainability pada Time Series Forecasting
## Reproduksi dari: [skforecast.org — Explainability](https://skforecast.org/0.15.1/user_guides/explainability.html)

---

## 📌 1. Analisa Prediksi Tentang Apa?

Tutorial ini melakukan **prediksi permintaan listrik harian (Electricity Demand)** di negara bagian **Victoria, Australia**.

- **Target (output)**: Total konsumsi listrik per hari dalam satuan **MW (Megawatt)**
- **Tujuan utama**: Bukan hanya meramalkan, tetapi **menjelaskan mengapa model membuat prediksi tersebut** (*explainability*)
- **Dataset**: `vic_electricity` — data setengah-jam dari 2012–2015, diagregasi menjadi **harian**

### Teknik Explainability yang digunakan:
| # | Metode | Scope |
|---|---|---|
| 1 | Feature Importance (LightGBM) | Global |
| 2 | SHAP Summary Plot | Global |
| 3 | SHAP Force Plot | **Lokal** (per prediksi) |
| 4 | SHAP Dependence Plot | Global |
| 5 | Permutation Importance | Global |
| 6 | Partial Dependence Plot (PDP) | Global |

---
## ⚙️ Instalasi Library

Jalankan cell berikut **sekali saja** untuk menginstall semua dependensi yang dibutuhkan.

In [1]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "skforecast", "lightgbm", "shap", "matplotlib", "pandas", "scikit-learn",
    "-q"
])
print("✅ Instalasi selesai! Restart kernel jika baru pertama kali install.")

✅ Instalasi selesai! Restart kernel jika baru pertama kali install.


---
## 📦 Import Library

| Library | Fungsi |
|---|---|
| `pandas` | Manipulasi dan analisis data tabular |
| `matplotlib` | Visualisasi grafik |
| `lightgbm` | Model gradient boosting berbasis tree |
| `skforecast` | Framework forecasting time series berbasis sklearn |
| `shap` | Menghitung dan memvisualisasikan SHAP values |
| `sklearn.inspection` | Permutation importance & Partial Dependence Plot |

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # backend non-interaktif, aman di VSCode
import matplotlib.pyplot as plt
import shap

from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay
from lightgbm import LGBMRegressor
from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive

# Nonaktifkan progress bar LightGBM
import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print("✅ Semua library berhasil diimport!")
print(f"   skforecast  : {__import__('skforecast').__version__}")
print(f"   shap        : {shap.__version__}")
print(f"   lightgbm    : {__import__('lightgbm').__version__}")

✅ Semua library berhasil diimport!
   skforecast  : 0.22.0
   shap        : 0.52.0
   lightgbm    : 4.6.0


---
## 📊 2. Bentuk Data Training

### Sumber Data
Dataset `vic_electricity` dari paket `tsibbledata` (R), berisi konsumsi listrik Victoria, Australia.

### Kolom setelah agregasi harian:
| Kolom | Peran | Keterangan |
|---|---|---|
| `Demand` | **Target (y)** | Total konsumsi listrik per hari (sum) |
| `Temperature` | **Fitur eksogen** | Suhu rata-rata per hari (mean) |

In [3]:
# Download dan load dataset
data = fetch_dataset(name="vic_electricity")
print(f"Shape data asli : {data.shape}")
print("\n5 baris pertama:")
print(data.head().to_string())

╭──────────────────────────── vic_electricity ─────────────────────────────╮
│ Description:                                                             │
│ Half-hourly electricity demand for Victoria, Australia                   │
│                                                                          │
│ Source:                                                                  │
│ O'Hara-Wild M, Hyndman R, Wang E, Godahewa R (2022).tsibbledata: Diverse │
│ Datasets for 'tsibble'. https://tsibbledata.tidyverts.org/,              │
│ https://github.com/tidyverts/tsibbledata/.                               │
│ https://tsibbledata.tidyverts.org/reference/vic_elec.html                │
│                                                                          │
│ URL:                                                                     │
│ https://raw.githubusercontent.com/skforecast/skforecast-                 │
│ datasets/main/data/vic_electricity.csv                                   │
│                                                                          │
│ Shape: 52608 rows x 4 columns                                            │
╰──────────────────────────────────────────────────────────────────────────╯

Shape data asli : (52608, 4)

5 baris pertama:
                          Demand  Temperature        Date  Holiday
Time                                                              
2011-12-31 13:00:00  4382.825174        21.40  2012-01-01     True
2011-12-31 13:30:00  4263.365526        21.05  2012-01-01     True
2011-12-31 14:00:00  4048.966046        20.70  2012-01-01     True
2011-12-31 14:30:00  3877.563330        20.55  2012-01-01     True
2011-12-31 15:00:00  4036.229746        20.40  2012-01-01     True


In [4]:
# Agregasi dari setengah-jam → harian
data = data.resample('D').agg({'Demand': 'sum', 'Temperature': 'mean'})

print(f"Shape setelah agregasi: {data.shape}")
print(f"Periode data: {data.index.min().date()} s.d. {data.index.max().date()}")
print("\n5 baris pertama:")
print(data.head().to_string())

Shape setelah agregasi: (1097, 2)
Periode data: 2011-12-31 s.d. 2014-12-31

5 baris pertama:
                   Demand  Temperature
Time                                  
2011-12-31   82531.745918    21.047727
2012-01-01  227778.257304    26.578125
2012-01-02  275490.988882    31.751042
2012-01-03  258955.329422    24.567708
2012-01-04  213792.376946    18.191667


In [5]:
# Visualisasi time series
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(data.index, data['Demand'], color='steelblue', linewidth=0.8)
axes[0].set_title('Konsumsi Listrik Harian — Victoria, Australia', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Demand (MW)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(data.index, data['Temperature'], color='tomato', linewidth=0.8)
axes[1].set_title('Suhu Rata-rata Harian — Melbourne', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Temperature (°C)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## ❓ 3. Apa Itu Lag?

**Lag** adalah nilai dari **waktu sebelumnya** yang dijadikan sebagai fitur input.

Model ML seperti LightGBM **tidak mengerti urutan waktu** secara otomatis.  
Dengan lag, kita mengubah time series menjadi **regresi tabular biasa**.

### Ilustrasi dengan `lags = 7`:

```
Hari ini (Senin) → ingin diprediksi
  lag_1 = demand kemarin       (Minggu)
  lag_2 = demand 2 hari lalu   (Sabtu)
  lag_3 = demand 3 hari lalu   (Jumat)
  ...      ...
  lag_7 = demand 7 hari lalu   (Senin minggu lalu)
```

### Bentuk matriks training:

| lag_1 | lag_2 | ... | lag_7 | Temperature | **y (target)** |
|---|---|---|---|---|---|
| 205338 | 211066 | ... | 82531 | 24.09 | **200693** |
| 200693 | 205338 | ... | 227778 | 20.22 | **200061** |

---
## 🤖 Membuat dan Melatih Forecaster

`ForecasterRecursive` dari skforecast secara otomatis:
1. Membuat fitur lag dari time series
2. Melatih model LightGBM dengan fitur tersebut
3. Saat prediksi, output sebelumnya dipakai sebagai input (recursive/autoregressive)

In [6]:
# Buat dan latih forecaster
forecaster = ForecasterRecursive(
    regressor = LGBMRegressor(random_state=123, verbose=-1),
    lags = 7
)

forecaster.fit(
    y    = data['Demand'],
    exog = data['Temperature']
)

print("✅ Forecaster berhasil dilatih!")
print(forecaster)

TypeError: ForecasterRecursive.__init__() got an unexpected keyword argument 'regressor'

In [ ]:
# Buat matriks training (X_train, y_train)
X_train, y_train = forecaster.create_train_X_y(
    y    = data['Demand'],
    exog = data['Temperature']
)

print("=" * 60)
print("MATRIKS X_train (INPUT FITUR):")
print("=" * 60)
print(f"Shape  : {X_train.shape}  ({X_train.shape[0]} baris × {X_train.shape[1]} kolom)")
print(f"Kolom  : {X_train.columns.tolist()}")
print("\n5 baris pertama:")
print(X_train.head().to_string())

print("\n" + "=" * 60)
print("y_train (OUTPUT / TARGET):")
print("=" * 60)
print(y_train.head().to_string())

---
## 🧪 4. Proses Analisis Explainability

Ada **6 tahap** analisis untuk memahami perilaku model:

| Tahap | Metode | Pertanyaan yang dijawab |
|---|---|---|
| 1 | Feature Importance | Fitur mana yang paling sering dipakai model? |
| 2 | SHAP Summary Plot | Fitur mana yang paling berpengaruh & ke arah mana? |
| 3 | SHAP Force Plot | Mengapa model membuat prediksi X pada tanggal Y? |
| 4 | SHAP Dependence Plot | Bagaimana hubungan fitur tertentu dengan prediksi? |
| 5 | Permutation Importance | Jika fitur ini dihilangkan, seberapa buruk modelnya? |
| 6 | Partial Dependence Plot | Apa efek rata-rata fitur terhadap target? |

### 📊 Tahap 1: Model-Specific Feature Importance

LightGBM menghitung kepentingan fitur berdasarkan **berapa kali fitur digunakan untuk split**  
di semua decision tree dalam ensemble. Semakin sering dipakai = semakin penting.

skforecast menyediakan `get_feature_importances()` untuk mengaksesnya langsung.

In [ ]:
# Ambil feature importance
importance = forecaster.get_feature_importances()
importance_sorted = importance.sort_values('importance', ascending=False)

print("Feature Importance (diurutkan):")
print(importance_sorted.to_string(index=False))

# Visualisasi
fig, ax = plt.subplots(figsize=(8, 5))
imp_plot = importance_sorted.sort_values('importance', ascending=True)
bars = ax.barh(imp_plot['feature'], imp_plot['importance'],
               color='steelblue', edgecolor='white', height=0.6)
ax.set_title('Feature Importance — LightGBM\n(Jumlah split di decision tree)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(True, axis='x', alpha=0.3)
for bar, val in zip(bars, imp_plot['importance']):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('plot_02_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_02_feature_importance.png")

---
### 📊 Tahap 2: SHAP Values — Summary Plot

**SHAP (SHapley Additive exPlanations)** berasal dari teori permainan kooperatif.  
SHAP menghitung **kontribusi marginal** setiap fitur terhadap prediksi.

#### Kelebihan SHAP:
- Bisa global (semua data) maupun **lokal** (satu prediksi)
- Menampilkan **arah pengaruh** (positif/negatif)
- Konsisten dan adil secara matematis

#### Membaca SHAP Summary Plot:
- **Sumbu Y** → Fitur
- **Sumbu X** → Nilai SHAP (+ = dorong prediksi naik, − = turun)
- **Warna titik** → Nilai fitur (🔴 tinggi, 🔵 rendah)

In [ ]:
# Buat SHAP TreeExplainer
explainer   = shap.TreeExplainer(forecaster.regressor)
shap_values = explainer.shap_values(X_train)

print(f"Shape SHAP values : {shap_values.shape}")
print(f"Shape X_train     : {X_train.shape}")
print("\nSetiap baris = satu observasi, setiap kolom = nilai SHAP untuk satu fitur")

In [ ]:
# SHAP Summary Plot — Bar (global importance)
fig = plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_train, plot_type="bar", show=False)
plt.title('SHAP Summary Plot (Bar)\nRata-rata |SHAP value| per fitur', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_03_shap_bar.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_03_shap_bar.png")

In [ ]:
# SHAP Summary Plot — Beeswarm (distribusi + arah pengaruh)
fig = plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_train, show=False)
plt.title('SHAP Beeswarm Plot\nDistribusi nilai SHAP (merah=nilai fitur tinggi, biru=rendah)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_04_shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_04_shap_beeswarm.png")

---
### 📊 Tahap 3: SHAP Force Plot — Penjelasan Prediksi Individual

**Force Plot** menjelaskan **satu prediksi tertentu** secara detail.

Cara membaca:
- **Base value** → Rata-rata prediksi model (baseline)
- **Output value** → Prediksi untuk observasi ini
- **Warna merah** → Fitur yang mendorong prediksi **naik**
- **Warna biru** → Fitur yang mendorong prediksi **turun**
- **Lebar bar** → Besarnya kontribusi fitur

> ⚠️ Untuk VSCode/Jupyter lokal, kita gunakan `matplotlib=True` agar render sebagai gambar statis.

In [ ]:
# Force plot observasi pertama — matplotlib mode (kompatibel VSCode)
print("📊 Force Plot — Observasi pertama")
print(f"Tanggal  : {X_train.index[0].date()}")
print(f"Prediksi : {explainer.expected_value + shap_values[0,:].sum():.0f} MW")
print(f"Base val : {explainer.expected_value:.0f} MW")
print("\nNilai fitur:")
print(X_train.iloc[[0]].to_string())
print()

shap.force_plot(
    explainer.expected_value,
    shap_values[0, :],
    X_train.iloc[0, :],
    matplotlib = True,   # ← wajib untuk VSCode / Jupyter non-browser
    show       = False
)
plt.title(f"Force Plot: {X_train.index[0].date()}", fontsize=11, pad=50)
plt.savefig('plot_05_force_single.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_05_force_single.png")

In [ ]:
# Waterfall plot — alternatif force plot yang lebih jelas di notebook statis
print("📊 Waterfall Plot — Kontribusi fitur untuk prediksi pertama")

shap_exp = shap.Explanation(
    values        = shap_values[0, :],
    base_values   = explainer.expected_value,
    data          = X_train.iloc[0, :].values,
    feature_names = X_train.columns.tolist()
)

fig = plt.figure(figsize=(9, 5))
shap.waterfall_plot(shap_exp, show=False)
plt.title(f"Waterfall Plot: {X_train.index[0].date()}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_06_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_06_waterfall.png")

---
### 📊 Tahap 4: SHAP Dependence Plot

Menunjukkan hubungan antara **nilai satu fitur** dengan **nilai SHAP-nya**.

- **Sumbu X** → Nilai fitur
- **Sumbu Y** → Nilai SHAP (kontribusi terhadap prediksi)
- **Warna** → Fitur lain yang paling berinteraksi (dipilih otomatis)

Berguna untuk melihat apakah hubungan bersifat **linear atau non-linear**.

In [ ]:
# Dependence plot — Temperature
fig, ax = plt.subplots(figsize=(8, 5))
shap.dependence_plot("Temperature", shap_values, X_train, ax=ax, show=False)
ax.set_title("SHAP Dependence Plot: Temperature\n(Warna = fitur yang paling berinteraksi)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_07_dep_temperature.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_07_dep_temperature.png")

In [ ]:
# Dependence plot — lag_1
fig, ax = plt.subplots(figsize=(8, 5))
shap.dependence_plot("lag_1", shap_values, X_train, ax=ax, show=False)
ax.set_title("SHAP Dependence Plot: lag_1 (Demand kemarin)\n(Warna = fitur yang paling berinteraksi)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_08_dep_lag1.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_08_dep_lag1.png")

---
### 📊 Tahap 5: Permutation Feature Importance

Metode **model-agnostik** yang mengukur pentingnya fitur dengan cara:
1. Latih model, catat error baseline
2. **Acak nilai fitur tertentu** secara random
3. Hitung peningkatan error
4. Semakin besar peningkatan error = semakin penting fitur tersebut

$$I_i = \text{Error}_{\text{permuted}(i)} - \text{Error}_{\text{original}}$$

| | Feature Importance | Permutation Importance |
|---|---|---|
| Basis | Struktur tree | Perubahan performa |
| Model-agnostik | ❌ | ✅ |
| Bias fitur banyak nilai | Ya | Tidak |

In [ ]:
# Hitung Permutation Importance
result = permutation_importance(
    estimator    = forecaster.regressor,
    X            = X_train,
    y            = y_train,
    n_repeats    = 5,
    random_state = 123,
    n_jobs       = -1
)

perm_df = pd.DataFrame({
    'feature'        : X_train.columns,
    'importance_mean': result.importances_mean,
    'importance_std' : result.importances_std
}).sort_values('importance_mean', ascending=False)

print("Permutation Importance:")
print(perm_df.to_string(index=False))

In [ ]:
# Visualisasi Permutation Importance
fig, ax = plt.subplots(figsize=(8, 5))
sorted_df = perm_df.sort_values('importance_mean', ascending=True)
ax.barh(
    sorted_df['feature'],
    sorted_df['importance_mean'],
    xerr      = sorted_df['importance_std'],
    color     = 'darkorange',
    alpha     = 0.85,
    error_kw  = dict(ecolor='black', capsize=4, linewidth=1.2)
)
ax.set_title('Permutation Feature Importance\n(Mean ± Std dari 5 pengulangan)', fontsize=13, fontweight='bold')
ax.set_xlabel('Peningkatan Error saat fitur diacak')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('plot_09_permutation_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_09_permutation_importance.png")

---
### 📊 Tahap 6: Partial Dependence Plot (PDP)

**PDP** menampilkan efek **rata-rata** satu fitur terhadap prediksi,  
dengan cara memarginalisasi semua fitur lainnya.

- `kind='average'` → Hanya garis rata-rata (PDP murni)
- `kind='both'` → PDP + Individual Conditional Expectation (ICE) per observasi

Garis **ICE** (tipis) menunjukkan keragaman antar observasi — jika ICE berbeda-beda arah,  
ada **interaksi antar fitur** yang perlu diperhatikan.

In [ ]:
# Partial Dependence Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

PartialDependenceDisplay.from_estimator(
    estimator       = forecaster.regressor,
    X               = X_train,
    features        = ["Temperature", "lag_1"],
    kind            = 'both',
    ax              = axes,
    grid_resolution = 50,
    line_kw         = {"color": "steelblue", "linewidth": 2},
    ice_lines_kw    = {"color": "gray", "alpha": 0.1, "linewidth": 0.5}
)

axes[0].set_title('PDP: Temperature\n(Garis tebal=rata-rata, tipis=individual)', fontsize=11)
axes[1].set_title('PDP: lag_1 (Demand kemarin)\n(Garis tebal=rata-rata, tipis=individual)', fontsize=11)
for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Partial Dependence Plot', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_10_pdp.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan: plot_10_pdp.png")

---
## ✅ Ringkasan Temuan Analisis

### Fitur Paling Penting
Berdasarkan semua metode explainability:

| Fitur | Pengaruh | Alasan |
|---|---|---|
| **Temperature** | ⭐ Terbesar | Suhu dingin → pemanas, panas → AC |
| **lag_1** | ⭐⭐ | Konsumsi kemarin sangat relevan (momentum) |
| **lag_7** | ⭐ | Pola mingguan (hari kerja vs weekend) |

### Insight dari Setiap Plot

1. **Feature Importance** → Temperature dan lag_1 dominan
2. **SHAP Beeswarm** → Temperature tinggi (merah) → SHAP tinggi (dorong prediksi naik)
3. **Force Plot** → Per prediksi, bisa dilihat fitur mana yang dominan mendorong nilai
4. **Dependence Plot** → Hubungan Temperature–Demand berbentuk **U** (non-linear)
5. **Permutation Imp.** → Konsisten dengan SHAP, Temperature paling krusial
6. **PDP** → Semakin tinggi lag_1 → semakin tinggi prediksi demand

---

## 📚 Kesimpulan

> **Explainability** bukan hanya akademis — dalam aplikasi nyata (energi, kesehatan, keuangan),  
> memahami *mengapa* model membuat prediksi tertentu sangat krusial untuk **kepercayaan dan pengambilan keputusan**.

| Metode | Scope | Kelebihan Utama |
|---|---|---|
| Feature Importance | Global | Cepat, bawaan model |
| Permutation Importance | Global | Model-agnostik, lebih objektif |
| SHAP Summary (bar) | Global | Rata-rata kontribusi absolut |
| SHAP Beeswarm | Global | Arah + distribusi pengaruh |
| SHAP Force/Waterfall | **Lokal** | Penjelasan per-prediksi |
| SHAP Dependence | Global | Non-linearitas + interaksi |
| PDP + ICE | Global | Efek marginal fitur terhadap target |